# A quarter car on a shaker

One corner of a car, reduced to its vertical motion. The body's share of the mass (the sprung mass) rides on a spring and a damper; below it are the wheel and its share of the axle (the unsprung mass); below those is the tyre, a stiff spring that can only push. The road is a plate driven up and down, as on a shaker rig, so any road profile can be fed in. The notebook checks:

1. the heights at rest, from the weights and the springs;
2. the two modes, body bounce and wheel hop, with their frequencies, damping and shapes, against the roots of the characteristic equation;
3. the response to the shaker's sine at frequencies from 0.5 to 20 Hz, against the closed-form transmissibility;
4. a bump that throws the wheel off the road: while the wheel flies, gravity alone acts on the car, and its momentum must fall at exactly (m_s + m_u) g.

It plays the modes and the bump back in Rerun.

Each check prints a result beside its closed form and stops the notebook if they differ by more than the allowance derived beside it. `python/tests/test_notebooks.py` runs the notebook in the same way, so these numbers are tested. (Plan task 6.6.)

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import multibody as mbd


def check(what, value, expected, allowed):
    """Print a result beside the value it should have; stop if they differ by more than `allowed`."""
    difference = float(np.max(np.abs(np.asarray(value) - np.asarray(expected))))
    if np.ndim(value) == 0:
        print(f"{what}: {value:.12g}, expected {expected:.12g}, difference {difference:.1e} (allowed {allowed:.1e})")
    else:
        print(f"{what}: largest difference {difference:.1e} (allowed {allowed:.1e})")
    assert difference <= allowed, what

## The model

The masses, springs and damper of the kernel's quarter-car test (`tests/test_quarter_car.cpp`): a sprung mass of 250 kg, an unsprung mass of 40 kg, a suspension spring of 20 kN/m (free length 0.4 m) with a damper of 1.5 kN s/m, and a tyre of 200 kN/m and radius 0.3 m, without damping. Z is up, as in ISO 8855, and gravity −Z.

Three bodies, each on a prismatic joint along Z from the ground, so that each coordinate is a height: the road plate, the wheel's centre and the body. A driver makes the plate's height follow the road's profile z_r(t). The suspension is a spring-damper between the wheel's origin and the body's. The tyre is a contact sphere of radius 0.3 m on the wheel, against the plate's top face, with exponent 1: it pushes with k_t δ when it penetrates by δ, and not at all when apart.

The body's centre of mass is set 0.35 m above its frame, which only moves where its box is drawn: the joints allow no rotation, so the offset changes no equation.

In [ ]:
m_s, m_u = 250.0, 40.0                         # sprung and unsprung masses [kg]
k_s, c_s, free_length = 20e3, 1.5e3, 0.4       # suspension [N/m], [N s/m], [m]
k_t, radius = 200e3, 0.3                       # tyre [N/m], [m]
g = mbd.g_accel


def quarter_car(road):
    """The quarter car on a plate whose height follows road = (z, dz/dt, d2z/dt2), three functions of time."""
    system = mbd.System()
    model = system.model
    model.gravity = [0.0, 0.0, -g]
    slide = mbd.PrismaticJointModel()                          # along Z: each coordinate is a height
    plate = mbd.RigidBodyInertia.from_solid_box(100.0, [0.4, 0.4, 0.02])
    plate.com_B = [0.0, 0.0, -0.02]                            # its top face at its frame, where the tyre touches
    body = mbd.RigidBodyInertia.from_solid_box(m_s, [0.6, 0.45, 0.15])
    body.com_B = [0.0, 0.0, 0.35]                              # drawn above the wheel
    model.add_body("ground", slide, inertia=plate, name="road")
    model.add_body("ground", slide, inertia=mbd.RigidBodyInertia.from_solid_box(m_u, [0.25, 0.1, 0.25]), name="wheel")
    model.add_body("ground", slide, inertia=body, name="body")
    system.add_constraint(mbd.JointDriver(model, model.body("road"), mbd.TimeFunction(*road)))
    system.add_force_element(mbd.LinearSpringDamper(model.body("wheel"), model.body("body"), [0.0, 0.0, 0.0],
                                                    [0.0, 0.0, 0.0], k_s, c_s, free_length))
    system.add_force_element(mbd.PlaneContact(model.body("wheel"), [[0.0, 0.0, 0.0]], radius, model.body("road"),
                                              mbd.Transform3(), mbd.ContactParams(stiffness=k_t)))
    return system


flat = (lambda t: 0.0, lambda t: 0.0, lambda t: 0.0)

## At rest

On a flat road `static_equilibrium` finds where the forces balance. The tyre carries both weights and the spring the body's, so the wheel's centre sits (m_s + m_u) g / k_t below the tyre's radius, and the body m_s g / k_s below the spring's free length above the wheel. Statics stops when every acceleration is below 1e-6 m/s², which leaves the heights within 1e-6 / 8.7² ≈ 1e-8 m (8.7 rad/s is the slower mode, below): allowed 1e-7 m.

In [ ]:
sim = mbd.Simulator(quarter_car(flat))
sim.q[:] = [0.0, radius, radius + free_length]        # just touching, the spring at its free length
print(mbd.static_equilibrium(sim))
q_rest = sim.q.copy()
wheel_rest = radius - (m_s + m_u) * g / k_t
check("wheel's centre at rest [m]", q_rest[1], wheel_rest, 1e-7)
check("body at rest [m]", q_rest[2], wheel_rest + free_length - m_s * g / k_s, 1e-7)

## Body bounce and wheel hop

With the tyre on the road the model is linear. On the heights (z_s, z_u) of body and wheel, the masses are M = diag(m_s, m_u), the damping C = c_s [[1, −1], [−1, 1]] and the stiffness K = [[k_s, −k_s], [−k_s, k_s + k_t]]. A free motion e^{λt} needs det(λ²M + λC + K) = 0, a quartic:

$$m_s m_u \lambda^4 + c_s (m_s + m_u)\,\lambda^3 + \big(m_s (k_s + k_t) + m_u k_s\big)\,\lambda^2 + c_s k_t\,\lambda + k_s k_t = 0.$$

Its roots, found by NumPy, are compared below with `linearize`, which differentiates the engine's equations about the state at rest (the driver holding the plate). Each root gives a frequency |λ|, a damping ratio −Re λ / |λ|, and a shape: the wheel's motion per unit of the body's, (m_s λ² + c_s λ + k_s) / (c_s λ + k_s). The model is linear in its states, so the central differences are exact but for rounding: allowed 1e-8 relative.

In [ ]:
linear = mbd.linearize(sim)
print(linear)
roots = np.roots([m_s * m_u, c_s * (m_s + m_u), m_s * (k_s + k_t) + m_u * k_s, c_s * k_t, k_s * k_t])
roots = sorted(roots[roots.imag > 0.0], key=abs)                 # one of each pair, the slower first
for name, mode, root in zip(("body bounce", "wheel hop"), linear.modes, roots):
    print(f"{name}, {mode.natural_frequency / (2.0 * math.pi):.3f} Hz:")
    check("  frequency [rad/s]", mode.natural_frequency, abs(root), 1e-8 * abs(root))
    check("  damping ratio", mode.damping_ratio, -root.real / abs(root), 1e-8)
    wheel_per_body = mode.shape[1] / mode.shape[2]                # the shape is over (road, wheel, body)
    closed_form = (m_s * root**2 + c_s * root + k_s) / (c_s * root + k_s)
    check("  wheel per unit of body", wheel_per_body, closed_form, 1e-8 * abs(closed_form))

### The modes, animated

Each mode as a free motion, q = q_rest + Re(a · shape · e^{λt}), played in Rerun for five of its decay times: the body bouncing on its suspension at 1.4 Hz, and the wheel hopping between body and road at 11.6 Hz. The shapes are given in velocity coordinates, which here are the rates of the heights, so they serve for the heights too. Each replay is a recording of its own; the viewer lists them on its left.

In [ ]:
for name, mode, size in (("body bounce", linear.modes[0], 0.05), ("wheel hop", linear.modes[1], 0.01)):
    time = np.linspace(0.0, 5.0 / -mode.eigenvalue.real, 600)
    motion = q_rest + size * np.real(np.outer(np.exp(mode.eigenvalue * time), mode.shape))
    mbd.replay(sim.system, (time, motion), name=name)

## On the shaker: the response at each frequency

The plate moves as z_r = A sin ωt with A = 2 mm, small enough for the tyre never to leave it (its deflection at rest is 14 mm), so the model stays linear. Once the start's transient has died away, body and wheel move as sines of the same frequency, and their amplitudes divided by A are the transmissibilities. In closed form, with Z_s and Z_u the complex amplitudes of body and wheel:

$$\begin{bmatrix} k_s + i\omega c_s - \omega^2 m_s & -(k_s + i\omega c_s) \\ -(k_s + i\omega c_s) & k_s + k_t + i\omega c_s - \omega^2 m_u \end{bmatrix} \begin{bmatrix} Z_s \\ Z_u \end{bmatrix} = \begin{bmatrix} 0 \\ k_t A \end{bmatrix}.$$

Each run lasts 8 s with RK4 at a step h of 1 ms, and a sine, a cosine and a constant fitted by least squares to its last 2 s give the amplitude. By then the slower transient has decayed by e^{−2.55 × 6} ≈ 2e-7. RK4's own error is about (h ω)⁴ / 120 relative, with ω the larger of the forcing's frequency and the wheel hop's; ten times that is allowed.

In [ ]:
A, h = 0.002, 1e-3


def shaker(omega):
    """Body and wheel amplitudes per unit of the plate's, from a run; and the smallest tyre deflection seen."""
    road = (lambda t: A * math.sin(omega * t), lambda t: A * omega * math.cos(omega * t),
            lambda t: -A * omega**2 * math.sin(omega * t))
    sim = mbd.Simulator(quarter_car(road))
    sim.q[:] = q_rest
    sim.initialize()
    run = sim.run_recorded(8.0, h)
    t, q = run["time"], run["q"]
    late = t >= 6.0
    fit = np.column_stack([np.sin(omega * t[late]), np.cos(omega * t[late]), np.ones(np.count_nonzero(late))])
    amplitudes = [np.hypot(*np.linalg.lstsq(fit, q[late, k], rcond=None)[0][:2]) / A for k in (2, 1)]
    return amplitudes, np.min(radius - (q[:, 1] - q[:, 0]))


def transmissibility(omega):
    """Closed form: |Z_s| / A and |Z_u| / A."""
    suspension = k_s + 1j * omega * c_s
    Z = np.linalg.solve([[suspension - omega**2 * m_s, -suspension],
                         [-suspension, suspension + k_t - omega**2 * m_u]], [0.0, k_t])
    return np.abs(Z)


frequencies = np.geomspace(0.5, 20.0, 12)                          # [Hz]
runs = [shaker(2.0 * math.pi * f) for f in frequencies]
simulated = np.array([amplitudes for amplitudes, _ in runs])
deflection = min(smallest for _, smallest in runs)
print(f"smallest tyre deflection in any run: {1000.0 * deflection:.1f} mm")
assert deflection > 0.0, "the tyre left the plate, and the model was not linear"
relative = simulated / np.array([transmissibility(2.0 * math.pi * f) for f in frequencies]) - 1.0
allowed = 10.0 * (h * np.maximum(2.0 * math.pi * frequencies, linear.modes[1].natural_frequency)) ** 4 / 120.0
for f, (body, wheel), limit in zip(frequencies, relative, allowed):
    print(f"{f:6.2f} Hz: body {body:+.1e}, wheel {wheel:+.1e} (allowed {limit:.0e})")
assert np.all(np.abs(relative) <= allowed[:, None]), "a transmissibility differs from its closed form"

In [ ]:
fine = np.geomspace(0.3, 30.0, 400)
closed = np.array([transmissibility(2.0 * math.pi * f) for f in fine])
fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(fine, closed[:, 0], color="C0", label="body, closed form")
ax.loglog(fine, closed[:, 1], color="C1", label="wheel, closed form")
ax.loglog(frequencies, simulated[:, 0], "o", color="C0", label="body, simulated")
ax.loglog(frequencies, simulated[:, 1], "s", color="C1", label="wheel, simulated")
for mode in linear.modes:
    ax.axvline(mode.natural_frequency / (2.0 * math.pi), color="grey", ls="--", lw=0.8)
ax.set_xlabel("shaker frequency [Hz]")
ax.set_ylabel("amplitude / road amplitude")
ax.legend()
ax.grid(True, which="both", lw=0.3)

## Over a bump

Now a bump 5 cm high and 0.5 m long, a raised cosine, crossed at 10 m/s, so that the plate rises and falls in 50 ms. The wheel is thrown upwards, and after the crest the plate drops away faster than the suspension can pull the wheel down: the wheel leaves it.

While it flies, the only force from outside on wheel and body together is gravity; the suspension's forces act between them and cancel. Their momentum m_s ż_s + m_u ż_u must therefore fall at exactly (m_s + m_u) g. A Runge-Kutta method keeps every linear relation of its equations: in each step with no tyre force at any stage, the momentum falls by (m_s + m_u) g h, to rounding. The check runs from the first sample in the air to the last, so that every step in it begins and ends in the air. Rounding over its 55 steps, four stages each, on a momentum of a few hundred kg m/s, comes to some 55 × 4 × 1e-16 × 300 ≈ 7e-12 kg m/s: allowed 1e-10 kg m/s.

In [ ]:
height, length, speed, start = 0.05, 0.5, 10.0, 0.2
nu = 2.0 * math.pi * speed / length                    # the cosine's angular frequency in time [rad/s]


def on_bump(t):
    return start <= t <= start + length / speed


def bump(t):
    return 0.5 * height * (1.0 - math.cos(nu * (t - start))) if on_bump(t) else 0.0


def bump_rate(t):
    return 0.5 * height * nu * math.sin(nu * (t - start)) if on_bump(t) else 0.0


def bump_acceleration(t):
    return 0.5 * height * nu**2 * math.cos(nu * (t - start)) if on_bump(t) else 0.0


sim = mbd.Simulator(quarter_car((bump, bump_rate, bump_acceleration)))
sim.q[:] = q_rest
sim.initialize()
run = sim.run_recorded(1.5, h)
t, q, v = run["time"], run["q"], run["v"]
tyre = k_t * np.maximum(radius - (q[:, 1] - q[:, 0]), 0.0)       # the tyre's force [N], from its deflection
air = np.flatnonzero(tyre == 0.0)
assert len(air) > 0 and np.all(np.diff(air) == 1), "one flight"
first, last = air[0], air[-1]
print(f"in the air from {t[first]:.3f} s to {t[last]:.3f} s")
momentum = m_s * v[:, 2] + m_u * v[:, 1]
check("momentum lost in the air [kg m/s]", momentum[first] - momentum[last],
      (m_s + m_u) * g * (t[last] - t[first]), 1e-10)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
top.plot(t, 1000.0 * q[:, 0], label="road")
top.plot(t, 1000.0 * (q[:, 1] - q_rest[1]), label="wheel")
top.plot(t, 1000.0 * (q[:, 2] - q_rest[2]), label="body")
top.set_ylabel("height from rest [mm]")
top.legend()
top.grid(True)
bottom.plot(t, tyre / 1000.0, color="C3")
bottom.axvspan(t[first], t[last], color="grey", alpha=0.2, label="wheel in the air")
bottom.set_xlabel("time [s]")
bottom.set_ylabel("tyre force [kN]")
bottom.set_xlim(0.0, 1.0)
bottom.legend()
bottom.grid(True)

The bump in Rerun, the tyre's force plotted beside it. The crossing takes 50 ms, so slow the playback down in the viewer's time panel, or drag the time line.

In [ ]:
channels = pd.DataFrame({"tyre force [N]": tyre}, index=pd.Index(t, name="time"))
mbd.replay(sim.system, run, name="quarter car over a bump", channels=channels);